# Video Inference 
Step 1 - Load the model (notebook is self contained)

Why self-contained instead of importing from the other notebook: Jupyter notebooks can't easily "import" cells from another .ipynb directly — the clean way to share logic across notebooks is either duplicating the small setup block (what we're doing, simple and reliable) or converting shared logic into a proper .py module in your models/ folder (which we'll actually do when we build the master notebook, since that's when consolidation genuinely pays off).

Config - loads project paths once. Works from any machine/clone location; only `config.py` needs to exist at the repo root.

In [9]:
from config import BASE_DIR, DATASETS_DIR, DATASETS_FACESWAP_DIR, SAVED_MODELS_DIR, OUTPUTS_DIR, REPORTS_DIR


In [10]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from PIL import Image
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(weights=None)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)
model.load_state_dict(torch.load(str(SAVED_MODELS_DIR / "efficientnet_best.pth"), map_location=device))
model = model.to(device)
model.eval()

class_names = ['fake', 'real']
IMG_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

print("Model loaded, ready for video inference.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True  # speeds up repeated fixed-size inference on GPU

Model loaded, ready for video inference.


Step 2 - Video Prediction function


Samples num_frames evenly across the video (same even-spacing logic as your original frame extraction) — analyzing the whole video would be slow and mostly redundant, since adjacent frames look nearly identical

Runs your existing single-image pipeline (face-crop → resize → normalize → predict) on each sampled frame

Averages probabilities across frames (rather than just taking the first frame's answer) — this is more robust, since a single blurry or oddly-angled frame won't dominate the verdict

Also computes a majority vote as a secondary check — if averaging and majority vote disagree, that's a signal the video had mixed/uncertain frame-level results, worth surfacing rather than hiding

Returns per-frame predictions too, so you can inspect which specific frames drove the final verdict — useful for debugging and for the eventual UI display

In [11]:
def predict_video(video_path, num_frames=5):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return {"error": "Could not read video"}

    frame_indices = [int(i * total_frames / num_frames) for i in range(num_frames)]

    input_tensors = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()
        if not success:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

        if len(faces) > 0:
            faces_sorted = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
            x, y, w, h = faces_sorted[0]
            face_crop = frame[y:y+h, x:x+w]
        else:
            face_crop = frame

        face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(face_rgb)
        input_tensors.append(transform(pil_img))

    cap.release()

    if len(input_tensors) == 0:
        return {"error": "No frames could be processed"}

    # Single batched forward pass instead of one call per frame
    batch = torch.stack(input_tensors).to(device)
    with torch.inference_mode():
        outputs = model(batch)
        probs_batch = torch.softmax(outputs, dim=1).cpu().numpy()

    frame_predictions = np.argmax(probs_batch, axis=1).tolist()
    frame_probs = list(probs_batch)

    avg_probs = np.mean(frame_probs, axis=0)
    final_pred_idx = np.argmax(avg_probs)
    majority_vote = max(set(frame_predictions), key=frame_predictions.count)

    return {
        "prediction": class_names[final_pred_idx],
        "confidence": round(float(avg_probs[final_pred_idx]) * 100, 2),
        "fake_probability": round(float(avg_probs[0]) * 100, 2),
        "real_probability": round(float(avg_probs[1]) * 100, 2),
        "frames_analyzed": len(frame_predictions),
        "per_frame_predictions": [class_names[p] for p in frame_predictions],
        "majority_vote": class_names[majority_vote]
    }

Step 3 - Test it 

In [12]:
raw_dir = DATASETS_DIR / "raw_deepfakes"
video_files = os.listdir(raw_dir)
if not video_files:
    raise FileNotFoundError(f"No videos found in {raw_dir}")
test_video = os.path.join(str(raw_dir), video_files[0])

Step 4 - Web live cam inference

cv2.VideoCapture(0) — opens your default webcam as a live video stream (not a file this time)

Runs in an infinite loop, reading one frame at a time, detecting a face, predicting, and drawing a colored bounding box + label directly onto the frame — green for real, red for fake

cv2.imshow(...) — opens a native OS window (not inside the Jupyter notebook itself) showing the live annotated feed — this is different from everything else we've done, since it's not rendering in the browser

cv2.waitKey(1) & 0xFF == ord('q') — checks if you pressed the 'q' key; if so, breaks the loop and closes cleanly

Important: this is a genuinely live, real-time prediction on every single frame — no averaging across multiple frames like the video function, since we need instant feedback. This is honestly a stress-test for your model on your own face, likely never seen anything like it in training (which used only manipulated celebrity/actor footage) — so don't be surprised if predictions flicker or seem less stable than the video test above; that's expected and worth noting as a real limitation in your documentation later

In [13]:
def run_webcam_inference():
    cap = cv2.VideoCapture(0)  # 0 = default webcam
    
    if not cap.isOpened():
        print("Could not access webcam.")
        return
    
    print("Webcam started. Press 'q' in the video window to quit.")
    
    while True:
        success, frame = cap.read()
        if not success:
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
        
        if len(faces) > 0:
            faces_sorted = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
            x, y, w, h = faces_sorted[0]
            face_crop = frame[y:y+h, x:x+w]
            
            face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(face_rgb)
            input_tensor = transform(pil_img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = model(input_tensor)
                probs = torch.softmax(output, dim=1)[0]
                pred_idx = torch.argmax(probs).item()
                confidence = probs[pred_idx].item() * 100
            
            label = class_names[pred_idx].upper()
            color = (0, 0, 255) if label == "FAKE" else (0, 255, 0)  # red for fake, green for real
            
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
            cv2.putText(frame, f"{label} ({confidence:.1f}%)", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        else:
            cv2.putText(frame, "No face detected", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)
        
        cv2.imshow("Live Deepfake Detection - Press 'q' to quit", frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

Step 6 - running web cam

In [14]:
run_webcam_inference()

Webcam started. Press 'q' in the video window to quit.
